In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from mllabs import Experimenter, Connector, ColSelector, ProgressSessionLogger, TqdmProgressSession
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector
from mllabs.filter import RandomFilter
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from mllabs.collector import MetricCollector, ModelAttrCollector

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

2026-04-17 10:10:03.773434: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data_path = Path('data')

dict_expr = {
    'Mulching_Used_n': (pl.col('Mulching_Used') == 'Yes').cast(pl.Int8),
    'Soil_25': (pl.col('Soil_Moisture') < 25).cast(pl.Int8),
    'Temp_30': (pl.col('Temperature_C') > 30).cast(pl.Int8),
    'Rain_300': (pl.col('Rainfall_mm') < 300).cast(pl.Int8),
    'Wind_10': (pl.col('Wind_Speed_kmh') > 10).cast(pl.Int8),
}

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
)

df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])

y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']
X_bin = ['Mulching_Used_n']
X_num_bin = ['Soil_25', 'Temp_30', 'Rain_300', 'Wind_10']
X_all = X_num + X_cat + X_bin + X_num_bin

y_repl = {'Low': 0, 'Medium': 1, 'High': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y).cast(pl.String).replace(y_weight).cast(pl.Float32)}
)

In [3]:
!rm -rf exp/fe

In [4]:
if not os.path.exists('exp/fe'):
    sss1 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.8)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    e_fe = Experimenter.create(
        df_train, 'exp/fe', sp = sss1, sp_v = sss2, splitter_params={'y': y}, 
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    e_fe.set_grp('clf', method = 'predict', role = 'head', edges = {'y': [(None, y)], 'sample_weight': [(None, 'sample_weight')]})
    e_fe.set_grp(
        'xgb', parent = 'clf', processor=xgb.XGBClassifier, adapter=XGBoostAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
    e_fe.add_collector(ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result'))
    e_fe.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'})
    e_fe.add_collector(
        ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result'))
    e_fe.set_grp(
        'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
        params={'early_stopping_rounds': 50, 'verbose': 0, 'random_state': 123, 'cat_features': ColSelector(col_type='category')})
    e_fe.add_collector(
        ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result'))
    e_fe.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
    e_fe.add_collector(
        ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result'))
    e_fe.set_grp('lr', parent='clf', processor=LogisticRegression)

    e_fe.set_grp('pre', role = 'stage', method='transform')
    e_fe.set_node('std', grp='pre', processor=StandardScaler, edges={'X': [(None, X_num)]})
    e_fe.set_node('ohe', grp='pre', processor=OneHotEncoder, edges={'X': [(None, X_cat)]}, params={'sparse_output': False})
    e_fe.build()
    
    y_edges = {'y': [(None, y)]}
    e_fe.add_collector(
        MetricCollector('bAcc', Connector(edges = y_edges, role = 'head'), slice(-1, None), balanced_accuracy_score, include_train = True))
    e_fe.add_collector(
        ModelAttrCollector('lgb_feature_importance', Connector(processor=lgb.LGBMClassifier, edges=y_edges), 'feature_importances'))
    e_fe.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_gain', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'gain'}))
    e_fe.add_collector(
        ModelAttrCollector(
            'xgb_feature_importance_cover', Connector(processor=xgb.XGBClassifier, edges=y_edges), 'feature_importances', params = {'importance_type': 'cover'}))
    e_fe.add_collector(
        ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_pvc'))
    e_fe.add_collector(
        ModelAttrCollector('cb_interaction', Connector(processor=cb.CatBoostClassifier, edges=y_edges), 'feature_importances_interaction'))
    e_fe.add_collector(
        ModelAttrCollector('lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef'))
else:
    e_fe = Experimenter.load('exp/fe', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

📁 Created directory: exp/fe
Building 2 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 2 node(s)


In [ ]:
e_fe.set_node('xgb_base', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('lgb_base', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin)]}, params={'n_estimators': 10000, 'gpu': None})
e_fe.set_node('cb_base', grp='cb', edges={'X': [(None, X_num + X_cat + ['Mulching_Used'])]}, params={'n_estimators': 10000, 'gpu': 'yes'})
e_fe.set_node('nn_base', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin)]}, params={'epochs': 200, 'gpu': 'yes'})
e_fe.set_node('lr_base', grp='lr', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin)]})
e_fe.exp(finalize=True, n_jobs=2, gpu_id_list=[0]) 

In [16]:
e_fe.get_collector('bAcc').get_metrics_agg(None)[0]

,test,train,valid
cb_base,0.965143,0.990831,0.964347
lgb_base,0.967129,0.994087,0.966176
lr_base,0.850281,0.849520,0.850366
nn_base,0.962241,0.972527,0.959794
xgb_base,0.967986,0.993891,0.966524
